# Phase 1 · 02 — Tool calling **the SOTA way** (OpenAI SDK)

In `01` we hand-built everything: a text protocol, a regex parser, the loop.
It worked, but parsing free-form text is fragile.

The state-of-the-art fix has two parts:

1. **The provider fine-tunes the model** specifically to emit tool calls, and
   returns them as **structured, typed fields** (`message.tool_calls`) — so
   there is nothing to hand-parse.
2. **The SDK** gives you clean objects instead of raw HTTP + JSON.

But notice: the **loop is identical** to the one you built. Ask → if tool calls,
run them and feed results back → repeat. Only the plumbing got nicer.

In [ ]:
# %pip install openai
import os, json
from openai import OpenAI

MODEL = "gpt-4o-mini"     # a small, cheap, tool-capable model

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY is not set. Set it and restart the kernel before the actual model call.")
else:
    print("✅ API key found.")

# Built lazily inside run_conversation() so the cells above the model call
# still run without a key.

### The tools — same plain functions as before

In [ ]:
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

AVAILABLE_TOOLS = {"add": add, "multiply": multiply}

### Describe the tools as a **structured schema**

In `01` we described tools in prose inside the system prompt. Here we hand the
SDK a JSON schema per tool. The model was trained to consume exactly this shape,
which is why its tool calls come back reliably structured.

In [ ]:
def number_tool(name, description):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"},
                },
                "required": ["a", "b"],
            },
        },
    }

tools = [
    number_tool("add", "Add two numbers and return the sum."),
    number_tool("multiply", "Multiply two numbers and return the product."),
]

### The loop — same shape as `01`, no hand-parsing

The only differences from your from-scratch loop:
`msg.tool_calls` is a **typed list** (no regex), and tool results go back with
`role: "tool"` + a `tool_call_id` that links each result to its request.

In [ ]:
def run_conversation(user_message):
    client = OpenAI()                      # needs OPENAI_API_KEY
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:            # no structured tool calls -> final answer
            return msg.content

        for call in msg.tool_calls:       # run each requested tool
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = AVAILABLE_TOOLS[name](**args)
            print(f"  🔧 {name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,   # links this result to the request
                "content": str(result),
            })

In [ ]:
answer = run_conversation("What is 12.5 times 8, and then add 100 to the result?")
print("\nFinal answer:", answer)

In [ ]:
answer = run_conversation("In one sentence, what is a tool call?")   # should call no tools
print("\nFinal answer:", answer)

## Recap: from-scratch vs SOTA

| | `01` from scratch | `02` SDK / SOTA |
|---|---|---|
| Tool request format | free text we invented | structured `tool_calls`, model fine-tuned for it |
| Parsing | our regex (brittle) | typed fields (robust) |
| Transport | raw HTTP | SDK objects |
| **The loop** | **ours** | **the same loop** |

The core idea never changed — the SDK just made it reliable and ergonomic.

**Next → `03_mcp_from_scratch.ipynb`:** so far the tools lived *inside* the
program. MCP lets you publish tools from a separate **server** that any client
can discover. We'll build that protocol from scratch too.